# Market Anomaly Detection

**Presented By:** Mohd Abuzar | **Company:** STARlab Capital | **Date:** 20th May, 2026

**Dataset:** [hourly MSFT stock data (2017-2026)](https://starlabcapital-my.sharepoint.com/:x:/p/haider/IQA_TaNlHafsSZLUF2z2GeYwAXMGNnXJwfG-3NcL09gvawI?e=nGiYT8)

## Sections
1. Dataset Overview
2. Missing Value Analysis
3. Data Quality Checks
4. Price Analysis (OHLC distributions, time series)
5. Bid-Ask Spread Analysis
6. VWAP Analysis
7. Return Distribution Analysis
8. Correlation Analysis
9. Stationarity Testing (ADF test)
10. Seasonality & Periodicity Patterns
11. Initial Outlier Detection
12. Summary of Key Findings

In [21]:
# Notebook imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import shap
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from pathlib import Path

In [22]:
# Notebook configuration
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 100,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11,
    "lines.linewidth": 1.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "price": "#2563eb",
    "volume": "#7c3aed",
    "anomaly": "#dc2626",
    "spread": "#059669",
    "normal": "#6b7280",
    "highlight": "#f59e0b",
}

plots_dir = Path(r"C:\Users\hh\Market-Anomaly-Detection\plots")
plots_dir.mkdir(parents=True, exist_ok=True)

CSV_PATH = Path("C:/Users/hh/Market-Anomaly-Detection/data/raw/msft_hourly(in).csv")
print("Setup complete. Libraries loaded.")

Setup complete. Libraries loaded.


In [23]:
# State check: preprocessing is the starting point
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def require_columns(frame, cols, name="df"):
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise RuntimeError(f"{name} missing columns: {', '.join(missing)}")

require(CSV_PATH.exists(), f"CSV not found at {CSV_PATH}. Update CSV_PATH in config.")
print("Preprocessing notebook: start here. No prerequisites.")


Preprocessing notebook: start here. No prerequisites.


In [24]:
# Imports and configuration moved to the top of the notebook.

---
## 1. Dataset Overview & Documentation

**Description:** Hourly OHLCV (Open, High, Low, Close, Volume) market data
for Microsoft Inc. (MSFT) stock, including bid/ask quotes and VWAP.

In [25]:
# Load raw data
df_raw = pd.read_csv(CSV_PATH)

print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\nColumns ({df_raw.shape[1]}):")
for i, col in enumerate(df_raw.columns, 1):
    dtype = df_raw[col].dtype
    non_null = df_raw[col].notna().sum()
    print(f"  {i:2d}. {col:25s}  {str(dtype):10s}  {non_null}/{len(df_raw)} non-null")

Shape: 15501 rows x 14 columns

Columns (14):
   1. Unnamed: 0                 int64       15501/15501 non-null
   2. underlying_symbol          str         15501/15501 non-null
   3. quote_datetime             str         15501/15501 non-null
   4. quote_date                 str         15501/15501 non-null
   5. quote_time                 str         15501/15501 non-null
   6. open                       float64     15501/15501 non-null
   7. high                       float64     15501/15501 non-null
   8. low                        float64     15501/15501 non-null
   9. close                      float64     15501/15501 non-null
  10. trade_volume               int64       15501/15501 non-null
  11. vwap                       float64     15501/15501 non-null
  12. bid                        float64     15501/15501 non-null
  13. ask                        float64     15501/15501 non-null
  14. mid                        float64     15501/15501 non-null


In [26]:
# Parse datetime and sort
if "df_raw" not in globals():
    raise RuntimeError("df_raw not found. Run the load-data cell first.")
require_columns(df_raw, ["quote_datetime", "open", "high", "low", "close"], name="df_raw")

df = df_raw.copy()
df["quote_datetime"] = pd.to_datetime(df["quote_datetime"], errors="coerce")
df = df.sort_values("quote_datetime").reset_index(drop=True)

# Drop unnamed index if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("DATASET DOCUMENTATION")
print(f"Source:          STARlab Capital provided dataset")
print(f"Asset:           MSFT (Microsoft Inc.)")
print(f"Time Range:      {df['quote_datetime'].min()} to {df['quote_datetime'].max()}")
print(f"Duration:        ~{(df['quote_datetime'].max() - df['quote_datetime'].min()).days / 365:.1f} years")
print(f"Frequency:       Hourly (with half-hour bar at 16:00 close)")
print(f"Total Rows:      {len(df):,}")
print(f"Trading Days:    {df['quote_datetime'].dt.date.nunique():,}")
print(f"\nKey Variables:")
print(f"  Price:    open, high, low, close")
print(f"  Quotes:   bid, ask, mid")
print(f"  Volume:   trade_volume (NOT used for anomaly detection per task spec)")
print(f"  VWAP:     vwap (volume-weighted average price)")

DATASET DOCUMENTATION
Source:          STARlab Capital provided dataset
Asset:           MSFT (Microsoft Inc.)
Time Range:      2017-01-03 10:30:00 to 2026-03-31 16:00:00
Duration:        ~9.2 years
Frequency:       Hourly (with half-hour bar at 16:00 close)
Total Rows:      15,501
Trading Days:    2,223

Key Variables:
  Price:    open, high, low, close
  Quotes:   bid, ask, mid
  Volume:   trade_volume (NOT used for anomaly detection per task spec)
  VWAP:     vwap (volume-weighted average price)


---
## 2. Descriptive Statistics

In [27]:
# Descriptive statistics for all numeric columns
if "df" not in globals():
    raise RuntimeError("df not found. Run the parsing cell first.")

print("\nDescriptive Statistics:")
df_numeric = df.select_dtypes(include=[np.number])  # filters numeric columns from df (irrespective of dtypes)
display_stats = df_numeric.describe().T
display_stats["missing"] = df_numeric.isnull().sum()
display_stats["missing_%"] = (df_numeric.isnull().sum() / len(df) * 100).round(2)
print("\n", display_stats.to_string())


Descriptive Statistics:

                 count          mean           std          min           25%           50%           75%           max  missing  missing_%
open          15501.0  2.434356e+02  1.283534e+02      62.1200  1.171350e+02  2.411000e+02  3.351800e+02  5.500000e+02        0        0.0
high          15501.0  2.442766e+02  1.287385e+02      62.2000  1.174300e+02  2.419900e+02  3.362300e+02  5.537200e+02        0        0.0
low           15501.0  2.425860e+02  1.279725e+02      61.9500  1.168500e+02  2.402900e+02  3.341400e+02  5.431700e+02        0        0.0
close         15501.0  2.434465e+02  1.283604e+02      62.1100  1.170800e+02  2.410700e+02  3.351700e+02  5.456100e+02        0        0.0
trade_volume  15501.0  3.147149e+06  2.180450e+06  514640.0000  1.797917e+06  2.559547e+06  3.776259e+06  4.256548e+07        0        0.0
vwap          15501.0  2.432814e+02  1.282347e+02      62.0702  1.170882e+02  2.411893e+02  3.350598e+02  5.443321e+02        0        0.0


---
## 3. Data Quality Checks

We check for:
- Invalid prices (negative, zero)
- OHLC consistency (high >= low, close in [low, high])
- Bid-ask consistency (bid <= ask)
- Duplicate timestamps
- Negative Volume
- Extreme price changes (> 20% in one bar)

In [28]:
print("DATA QUALITY CHECKS & FIXES")
if "df" not in globals():
    raise RuntimeError("df not found. Run the parsing cell first.")
require_columns(df, ["open", "high", "low", "close", "bid", "ask", "quote_datetime"], name="df")
print(f"Starting rows: {len(df)}")

# Keep chronological order before return-based checks
df = df.sort_values("quote_datetime").reset_index(drop=True)

# Check 1: Non-positive OHLC -> drop rows
price_cols = ["open", "high", "low", "close"]

DATA QUALITY CHECKS & FIXES
Starting rows: 15501


In [29]:
df

,underlying_symbol,quote_datetime,quote_date,quote_time,open,high,low,close,trade_volume,vwap,bid,ask,mid
0,MSFT,2017-01-03 10:30:00,1/3/2017,10:30:00,62.790,62.8400,62.4200,62.70,4678361,62.6543,62.69,62.70,62.695
1,MSFT,2017-01-03 11:30:00,1/3/2017,11:30:00,62.700,62.7600,62.2500,62.38,2674749,62.4188,62.37,62.38,62.375
2,MSFT,2017-01-03 12:30:00,1/3/2017,12:30:00,62.370,62.4800,62.2535,62.37,1758058,62.3656,62.37,62.38,62.375
3,MSFT,2017-01-03 13:30:00,1/3/2017,13:30:00,62.375,62.4200,62.2850,62.37,1359954,62.3566,62.36,62.37,62.365
4,MSFT,2017-01-03 14:30:00,1/3/2017,14:30:00,62.370,62.3950,62.2100,62.22,1713371,62.2879,62.21,62.22,62.215
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15496,MSFT,2026-03-31 12:30:00,3/31/2026,12:30:00,364.200,365.1499,363.6500,365.05,2100495,364.4337,365.02,365.07,365.045
15497,MSFT,2026-03-31 13:30:00,3/31/2026,13:30:00,365.045,370.5700,364.2900,368.88,5724045,367.7088,368.82,368.92,368.870
15498,MSFT,2026-03-31 14:30:00,3/31/2026,14:30:00,368.860,370.1500,368.0400,369.83,2990991,369.0851,369.79,369.87,369.830
15499,MSFT,2026-03-31 15:30:00,3/31/2026,15:30:00,369.810,371.7000,369.4300,371.09,4442160,370.6311,371.09,371.15,371.120


In [30]:
# Ensure processed CSV exists (write it if missing)

processed_path = Path("C:/Users/hh/Market-Anomaly-Detection/data/processed/msft_hourly(in)_processed.csv")
if not processed_path.exists():
    df.to_csv(processed_path, index=False)
    print(f"Saved processed data to: {processed_path}")
else:
    print(f"Processed data already exists: {processed_path}")


Processed data already exists: C:\Users\hh\Market-Anomaly-Detection\data\processed\msft_hourly(in)_processed.csv
